In [1]:
import subprocess
import time as t
import os
import re
import zipfile
import json
from tqdm import tqdm
import shutil

from src import preproc as pp
from src import model_eval as mev 


import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from src import gcmhn_model as gcmhn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Full Data Preproc

In [2]:
clinical_tab = pd.read_csv('data/investigator_nacc72.csv', low_memory = False)

## Getting Class Labels

In [3]:
clinical_tab['label'] = clinical_tab.apply(pp.assign_label, axis = 1)

clinical_tab = clinical_tab[clinical_tab["label"].notna()].copy() # getting rid of na ones (ambig/comorbid)

# need int labels
clinical_tab["label"] = clinical_tab["label"].astype(int)

print(clinical_tab["label"].value_counts().sort_index())

label_translate = {
    'CN': 0,
    'MCI': 1,
    'AD': 2,
    'VaD': 3,
    'LBD': 4,
    'FTD': 5}


label
0    88526
1    11674
2    47746
3    14478
4     3499
5     6252
Name: count, dtype: int64


## Getting patient id and mri image path mappings

In [4]:
import os
print(os.getcwd())
mri_base_path = "/home/stu15/s9/dan7492/Courses/idai710/project/nacc_mri"

/home/stu15/s9/dan7492/Courses/idai710/project


In [5]:
# Getting dict mapping

pid_mri = {}

for pid in os.listdir(mri_base_path): # getting all folders in dir
    
    p_dir = os.path.join(mri_base_path, pid) # indiv px dir

    if not os.path.isdir(p_dir): # skip non folders
        continue

    img_path = pp.select_t1_scan(pid, mri_base_path) # get best scan for Px

    if img_path is not None and os.path.exists(img_path):
        pid_mri[pid] = img_path

print("Valid MRI Pxs:", len(pid_mri))

Valid MRI Pxs: 223


## Filtering the tabular dataframe where only patients with valid scans are present

In [6]:
# filtering tabular df --> only cont pts with valid scans

tab_filt = clinical_tab[clinical_tab["NACCID"].isin(pid_mri.keys())].copy()

tab_filt = tab_filt.reset_index(drop = True)

print("Px rows w both clin and MRI", len(tab_filt))
print(tab_filt["label"].value_counts().sort_index()) # gives more rows than Pxs, so likely multiple visits

# Now going to keep first visit only for simplification
tab_filt = tab_filt.sort_values(['NACCID', 'VISITYR', 'VISITMO'])
tab_filt = tab_filt.drop_duplicates(subset = "NACCID", keep = 'first').reset_index(drop=True)
print("=== === === ===")
print("Px rows w both clin and MRI", len(tab_filt))
print(tab_filt["label"].value_counts().sort_index()) # now is good

Px rows w both clin and MRI 1213
label
0    360
1    121
2    184
3    258
4    103
5    187
Name: count, dtype: int64
=== === === ===
Px rows w both clin and MRI 223
label
0    68
1    20
2    43
3    38
4    16
5    38
Name: count, dtype: int64


## Defining some cols to use for tabular data, and defining a new col for 'statin' use
> Medications called statins (atorvastatin, rosuvastatin, etc.) used to lower cholesterol have an increased risk of dementia

In [7]:
# Getting tabular input cols to use --> also defining ones for statin use since associated w/ dementia risk

med_col_idxs = [i for i in range(75,115)] # col ixd 75-114 is for px meds, want to create binary cols for statin use 
rest_tabular = [ 'BIRTHYR', 'SEX', 'HISPANIC', 'RACE', 'WEIGHT'] 

drug_cols = tab_filt.columns[75:115]

def is_statin(x):
    if pd.isna(x): # defining fn to use which searches for statin presence in drug cols and retruns 0 or 1
        return False
    x = str(x).lower()
    return 'statin' in x 

tab_filt['STATIN_USE'] = tab_filt[drug_cols].apply(
    lambda row: any(is_statin(x) for x in row), axis = 1
).astype(int)

print(tab_filt['STATIN_USE'].value_counts())

STATIN_USE
0    136
1     87
Name: count, dtype: int64


In [8]:
# Then filtering all the cols chosen for model input

full_cols = ['NACCID'] + rest_tabular + ['STATIN_USE'] + ['label']
tab_feat_cols = rest_tabular + ['STATIN_USE']

tab_filt_full = tab_filt[full_cols]

print(tab_filt_full.columns)

Index(['NACCID', 'BIRTHYR', 'SEX', 'HISPANIC', 'RACE', 'WEIGHT', 'STATIN_USE',
       'label'],
      dtype='object')


## Normalizing the continuous tabular features

In [9]:
# Now normalizing the tabular features
cont_cols = ['BIRTHYR', 'WEIGHT']

for col in cont_cols:
    mu = tab_filt_full[col].mean()
    std = tab_filt_full[col].std()

    tab_filt_full.loc[:, col] = (tab_filt_full[col] - mu) / (std + 1e-8)

print(tab_filt_full[cont_cols].mean())
print(tab_filt_full[cont_cols].std())

BIRTHYR   -7.997589e-15
WEIGHT     9.160585e-17
dtype: float64
BIRTHYR    1.0
WEIGHT     1.0
dtype: float64


/tmp/ipykernel_1033126/1463957939.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[-0.12722089 -0.57517234 -0.94846521 -0.35119661  2.2618535   1.06731631
 -0.27653804  0.69402343 -0.72448949  0.39538913 -0.42585519  0.17141341
 -1.02312379  0.09675483  1.29129203  1.06731631  1.06731631 -0.20187946
 -1.84436811  0.47004771 -1.62039239 -1.24709951  0.24607198 -0.42585519
 -1.24709951  0.91799916 -0.94846521 -0.35119661 -1.91902668 -0.72448949
 -0.79914806 -0.64983091 -0.87380664 -0.57517234 -0.72448949  0.39538913
  0.76868201  0.02209626 -0.50051376 -1.39641666  0.24607198 -0.12722089
  0.76868201  0.54470628 -0.64983091  0.47004771 -0.87380664 -0.42585519
 -0.27653804  0.32073056  3.30707355  0.76868201  1.21663346  0.09675483
  0.61936486 -0.20187946  0.99265773  0.61936486 -0.27653804  0.69402343
 -0.05256231 -0.05256231 -0.64983091  0.91799916 -1.69505096  0.84334058
  1.14197488  0.47004771 -0.64983091 -0

# Setting up the train/val/test process
# AND datasets/loaders AND model, loss, optimizer

In [11]:
train, val_test = train_test_split( # 70/10/20 train
    tab_filt_full, 
    test_size = 0.3,
    stratify = tab_filt_full["label"], # keeping balance across splis
    random_state = 1470
)

val, test = train_test_split(
    val_test,
    test_size=2/3,
    stratify=val_test["label"],
    random_state=1470
)

# getting separate nacc datasets for each split
train_ds = pp.nacc_mri_ds(train, tab_feat_cols, pid_mri)
val_ds = pp.nacc_mri_ds(val, tab_feat_cols, pid_mri)
test_ds = pp.nacc_mri_ds(test, tab_feat_cols, pid_mri)

# getting loaders for each
train_loader = DataLoader(train_ds, batch_size = 8, shuffle = True)
val_loader = DataLoader(val_ds, batch_size = 8, shuffle = False)
test_loader = DataLoader(test_ds, batch_size = 8, shuffle = False)

/home/stu15/s9/dan7492/miniforge3/envs/idai710/lib/python3.10/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [12]:
# Setting up model instantiation and optizer to use when updating params
model = gcmhn.GCMHN(clin_dim = 6, n_classes = 6).to(device) # have 6 tot disease classes, and 6 tabular features

# defining cross ent loss for loss calc --> now doing class weighted bc early results BAD
class_cnt = train["label"].value_counts().sort_index().values
class_cnt = torch.tensor(class_cnt, dtype = torch.float32)
class_wts = 1 / class_cnt
class_wts = class_wts / class_wts.sum() * len(class_wts)
class_wts = class_wts.to(device)

print("Counts:", class_cnt)
print("Weights", class_wts)

loss_crit = nn.CrossEntropyLoss(weight = class_wts)

# def adam optimizer
optimizer = optim.Adam(model.parameters(), lr = 5e-5)

Counts: tensor([47., 14., 30., 27., 11., 27.])
Weights tensor([0.4387, 1.4726, 0.6872, 0.7636, 1.8743, 0.7636], device='cuda:0')


# TRAINING LOOP

In [12]:
epochs = 50

best_val_f1 = 0

start = t.time()
for e in range(epochs):

    # TRAINING

    model.train() # putting model intrain mode so dropout and batch norm occur

    tr_loss = 0

    for img, tab, y in train_loader:

        img = img.to(device) # sending to cuda device
        tab = tab.to(device)
        y = y.to(device)

        optimizer.zero_grad() # out with old grad from prev batch

        # fw pass
        logits = model(img, tab)

        # comp clf loss
        loss = loss_crit(logits, y)

        # backprop

        loss.backward() # comp grads for learnable params

        optimizer.step() # update them

        tr_loss += loss.item() # storing loss val

    tr_loss = tr_loss / len(train_loader)

    # VAL

    val_loss, val_acc, val_f1, _, _, _ = mev.eval_model(model = model, loader = val_loader, loss_fn = loss_crit, device = device)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "best_model.pth")

    print(
        f"Epoch {e + 1}/{epochs} --- Train Loss: {tr_loss:.4f} --- Val Loss: {val_loss:.4f} --- Val Acc: {val_acc:.4f} --- Val Macro-F1: {val_f1:.4f}"
    )

end = t.time()

print(f"\n\n TOTAL TIME ELAPSED: {end - start}")

Epoch 1/50 --- Train Loss: 1.9570 --- Val Loss: 1.8462 --- Val Acc: 0.3182 --- Val Macro-F1: 0.0805
Epoch 2/50 --- Train Loss: 1.8217 --- Val Loss: 1.8556 --- Val Acc: 0.3182 --- Val Macro-F1: 0.0805
Epoch 3/50 --- Train Loss: 1.8244 --- Val Loss: 1.8234 --- Val Acc: 0.3182 --- Val Macro-F1: 0.0805
Epoch 4/50 --- Train Loss: 1.8133 --- Val Loss: 1.8055 --- Val Acc: 0.3182 --- Val Macro-F1: 0.0805
Epoch 5/50 --- Train Loss: 1.8749 --- Val Loss: 1.7993 --- Val Acc: 0.2727 --- Val Macro-F1: 0.1222
Epoch 6/50 --- Train Loss: 1.8336 --- Val Loss: 1.7918 --- Val Acc: 0.3182 --- Val Macro-F1: 0.1833
Epoch 7/50 --- Train Loss: 1.8666 --- Val Loss: 1.7960 --- Val Acc: 0.2727 --- Val Macro-F1: 0.1222
Epoch 8/50 --- Train Loss: 1.8638 --- Val Loss: 1.7877 --- Val Acc: 0.2727 --- Val Macro-F1: 0.1222
Epoch 9/50 --- Train Loss: 1.8126 --- Val Loss: 1.8066 --- Val Acc: 0.2727 --- Val Macro-F1: 0.1222
Epoch 10/50 --- Train Loss: 1.8386 --- Val Loss: 1.7917 --- Val Acc: 0.2727 --- Val Macro-F1: 0.1097

# TEST EVAL

In [13]:
model.load_state_dict(torch.load("best_model.pth", map_location = device))

test_loss, test_acc, test_f1, test_labs, test_preds, test_probs = mev.eval_model(
    model = model, loader = test_loader, loss_fn = loss_crit, device = device
)

print("FINAL TEST RESULTS")
print("------------------")
print(f" Test Loss: {test_loss:.4f}\n Test Acc: {test_acc:.4f}\n Test Macro-F1: {test_f1:.4f}")

FINAL TEST RESULTS
------------------
 Test Loss: 1.7947
 Test Acc: 0.2000
 Test Macro-F1: 0.1306


In [15]:
class_names = ['Normal', 'MCI', "Alzheimer's", 'Vascular', 'Lewy Body', 'Frontotemporal']

mev.plot_soft_prob_cf(true_labels = test_labs, predicted_probs = test_probs, class_names = class_names, 
                      plot_title = "GCMHN Soft Confusion Matrix")

TypeError: plot_soft_prob_cf() got an unexpected keyword argument 'pred_probs'